In [11]:
# ============================================================
# PHASE 4 — CELL 1
# Environment & Imports
# ============================================================

import os
import cv2
import json
import math
import time
import shutil
import traceback
import gc

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import mediapipe as mp


print("=" * 70)
print("PHASE 4 — MOUTH ROI EXTRACTION")
print("=" * 70)

print("Python      :", __import__("sys").version.split()[0])
print("NumPy       :", np.__version__)
print("OpenCV      :", cv2.__version__)
print("MediaPipe   :", mp.__version__)
print("=" * 70)

PHASE 4 — MOUTH ROI EXTRACTION
Python      : 3.11.9
NumPy       : 1.26.4
OpenCV      : 4.11.0
MediaPipe   : 0.10.14


In [12]:
# ============================================================
# PHASE 4 — CELL 2
# Production Configuration
# ============================================================

PROJECT_DIR = Path.cwd().resolve()

while PROJECT_DIR != PROJECT_DIR.parent:

    if (
        (PROJECT_DIR / "dataset").exists()
        and
        (PROJECT_DIR / "output").exists()
    ):
        break

    PROJECT_DIR = PROJECT_DIR.parent


OUTPUT_DIR = PROJECT_DIR / "output"

SUPPORTED_VIDEO_EXTENSIONS = {
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
    ".webm",
    ".mpg",
    ".mpeg",
}

# ------------------------------------------------------------
# Phase 4 output
# ------------------------------------------------------------

MOUTH_DIR_NAME = "mouth_crop"

MOUTH_SIZE = (96, 96)

JPEG_QUALITY = 95

# ------------------------------------------------------------
# MediaPipe
# ------------------------------------------------------------

MAX_NUM_FACES = 1

REFINE_LANDMARKS = True

MIN_DETECTION_CONFIDENCE = 0.5

MIN_TRACKING_CONFIDENCE = 0.5

# ------------------------------------------------------------
# Crop margin
# ------------------------------------------------------------

MOUTH_MARGIN_X = 20

MOUTH_MARGIN_Y = 20

# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------

FORCE_RERUN = False

SKIP_INVALID_FRAME = True

print("=" * 70)
print("PHASE 4 — CONFIGURATION")
print("=" * 70)

print("Project      :", PROJECT_DIR)
print("Output       :", OUTPUT_DIR)
print("Mouth size   :", MOUTH_SIZE)
print("MediaPipe    : Face Mesh")
print("Refine       :", REFINE_LANDMARKS)
print("Force rerun  :", FORCE_RERUN)
print("=" * 70)

PHASE 4 — CONFIGURATION
Project      : C:\LipReadingSSL
Output       : C:\LipReadingSSL\output
Mouth size   : (96, 96)
MediaPipe    : Face Mesh
Refine       : True
Force rerun  : False


In [13]:
# ============================================================
# PHASE 4 — CELL 3
# MediaPipe Face Mesh
# ============================================================

mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(

    static_image_mode=False,

    max_num_faces=MAX_NUM_FACES,

    refine_landmarks=REFINE_LANDMARKS,

    min_detection_confidence=
        MIN_DETECTION_CONFIDENCE,

    min_tracking_confidence=
        MIN_TRACKING_CONFIDENCE,
)

print("=" * 70)
print("MediaPipe Face Mesh Loaded")
print("=" * 70)

MediaPipe Face Mesh Loaded


In [14]:
# ============================================================
# PHASE 4 — CELL 4
# Mouth Landmark Definition
# ============================================================

MOUTH_LANDMARKS = [

    61,
    146,
    91,
    181,
    84,
    17,
    314,
    405,
    321,
    375,
    291,
    308,
    324,
    318,
    402,
    317,
    14,
    87,
    178,
    88,
    95,
    185,
    40,
    39,
    37,
    0,
    267,
    269,
    270,
    409,
    415,
    310,
    311,
    312,
    13,
    82,
    81,
    42,
    183,
    78,
]

print(
    "Mouth landmarks:",
    len(MOUTH_LANDMARKS)
)

Mouth landmarks: 40


In [15]:
# ============================================================
# PHASE 4 — CELL 5
# Path Helpers
# ============================================================

def get_video_id(video_path):

    return Path(video_path).stem


def get_phase4_paths(video_path):

    video_path = Path(video_path).resolve()

    video_id = get_video_id(video_path)

    video_output_dir = (
        OUTPUT_DIR / video_id
    )

    mouth_dir = (
        video_output_dir /
        MOUTH_DIR_NAME
    )

    metadata_path = (
        video_output_dir /
        "phase4_metadata.json"
    )

    stats_path = (
        video_output_dir /
        "phase4_stats.csv"
    )

    return {

        "video_id":
            video_id,

        "video_output_dir":
            video_output_dir,

        "mouth_dir":
            mouth_dir,

        "metadata":
            metadata_path,

        "stats":
            stats_path,
    }


print("Path helper ready")

Path helper ready


In [16]:
# ============================================================
# PHASE 4 — CELL 6
# Mouth Crop Function
# ============================================================

def crop_mouth(
    frame,
    landmarks,
    margin_x=MOUTH_MARGIN_X,
    margin_y=MOUTH_MARGIN_Y,
):

    if frame is None:
        return None

    if landmarks is None:
        return None

    h, w = frame.shape[:2]

    points = []

    for idx in MOUTH_LANDMARKS:

        if idx >= len(landmarks):
            continue

        x = int(
            landmarks[idx].x * w
        )

        y = int(
            landmarks[idx].y * h
        )

        points.append(
            [x, y]
        )

    if len(points) == 0:
        return None

    points = np.asarray(
        points,
        dtype=np.int32
    )

    x_min = max(
        0,
        int(points[:, 0].min())
        - margin_x
    )

    y_min = max(
        0,
        int(points[:, 1].min())
        - margin_y
    )

    x_max = min(
        w,
        int(points[:, 0].max())
        + margin_x
    )

    y_max = min(
        h,
        int(points[:, 1].max())
        + margin_y
    )

    if (
        x_max <= x_min
        or
        y_max <= y_min
    ):
        return None

    mouth = frame[
        y_min:y_max,
        x_min:x_max
    ]

    if mouth.size == 0:
        return None

    mouth = cv2.resize(
        mouth,
        MOUTH_SIZE,
        interpolation=cv2.INTER_AREA
    )

    return mouth

In [17]:
# ============================================================
# PHASE 4 — CELL 7
# Process One Aligned Face
# ============================================================

def process_aligned_frame(
    frame,
):

    if frame is None:

        return {
            "success": False,
            "reason": "invalid_frame",
            "mouth": None,
            "landmarks": None,
        }

    try:

        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        results = (
            face_mesh.process(rgb)
        )

    except Exception as e:

        return {
            "success": False,
            "reason":
                f"mediapipe_error:{type(e).__name__}",
            "mouth": None,
            "landmarks": None,
        }

    if (
        results is None
        or
        not results.multi_face_landmarks
    ):

        return {
            "success": False,
            "reason":
                "no_face_landmarks",
            "mouth": None,
            "landmarks": None,
        }

    face_landmarks = (
        results
        .multi_face_landmarks[0]
        .landmark
    )

    mouth = crop_mouth(
        frame,
        face_landmarks
    )

    if mouth is None:

        return {
            "success": False,
            "reason":
                "invalid_mouth_crop",
            "mouth": None,
            "landmarks":
                face_landmarks,
        }

    return {
        "success": True,
        "reason": "ok",
        "mouth": mouth,
        "landmarks":
            face_landmarks,
    }

In [18]:
# ============================================================
# PHASE 4 — CELL 8
# Discover Phase 3 Aligned Faces
# ============================================================

def find_aligned_face_dir(video_path):

    paths = get_phase4_paths(
        video_path
    )

    video_output_dir = (
        paths["video_output_dir"]
    )

    candidates = [

        video_output_dir /
        "aligned_faces",

        video_output_dir /
        "aligned_face",

        video_output_dir /
        "aligned",

    ]

    for path in candidates:

        if path.exists() and path.is_dir():

            return path

    return None


def discover_video_ids():

    video_dirs = []

    if not OUTPUT_DIR.exists():

        return []

    for path in OUTPUT_DIR.iterdir():

        if not path.is_dir():
            continue

        if (
            path.name.startswith(".")
            or
            path.name == "phase2_logs"
        ):
            continue

        video_dirs.append(
            path
        )

    video_dirs.sort(
        key=lambda p: p.name.lower()
    )

    return video_dirs


VIDEO_OUTPUT_DIRS = (
    discover_video_ids()
)

print(
    "Found output directories:",
    len(VIDEO_OUTPUT_DIRS)
)

for path in VIDEO_OUTPUT_DIRS:

    print(
        " -",
        path.name
    )

Found output directories: 4
 - dataset
 - video001
 - video002
 - video003


In [21]:
# ============================================================
# PHASE 4 — CELL 9
# Process One Video
# ============================================================

def process_video(
    video_output_dir
):

    video_output_dir = (
        Path(video_output_dir).resolve()
    )

    video_id = (
        video_output_dir.name
    )

    aligned_dir = (
        video_output_dir /
        "aligned_face"
    )

    mouth_dir = (
        video_output_dir /
        MOUTH_DIR_NAME
    )

    mouth_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    if not aligned_dir.exists():

        raise FileNotFoundError(
            f"Phase 3 aligned_faces not found:\n"
            f"{aligned_dir}"
        )

    image_files = [

        p

        for p in aligned_dir.iterdir()

        if (
            p.is_file()
            and
            p.suffix.lower()
            in {
                ".jpg",
                ".jpeg",
                ".png",
                ".bmp",
                ".webp",
            }
        )
    ]

    image_files.sort(
        key=lambda p: p.name
    )

    stats = {

        "video_id":
            video_id,

        "input_frames":
            len(image_files),

        "mouth_saved":
            0,

        "skipped":
            0,

        "no_landmarks":
            0,

        "invalid_crop":
            0,

        "errors":
            0,
    }

    print()
    print("=" * 70)
    print(
        f"PHASE 4 — {video_id}"
    )
    print("=" * 70)

    for image_path in tqdm(
        image_files,
        desc=video_id,
        unit="frame"
    ):

        frame_name = (
            image_path.stem
        )

        output_path = (
            mouth_dir /
            f"{frame_name}.jpg"
        )

        # ----------------------------------------------------
        # Existing output
        # ----------------------------------------------------

        if (
            output_path.exists()
            and
            not FORCE_RERUN
        ):

            stats["mouth_saved"] += 1

            continue

        # ----------------------------------------------------
        # Read image
        # ----------------------------------------------------

        frame = cv2.imread(
            str(image_path)
        )

        if frame is None:

            stats["skipped"] += 1

            continue

        # ----------------------------------------------------
        # MediaPipe
        # ----------------------------------------------------

        result = process_aligned_frame(
            frame
        )

        if not result["success"]:

            reason = result["reason"]

            stats["skipped"] += 1

            if reason == "no_face_landmarks":

                stats["no_landmarks"] += 1

            elif reason == "invalid_mouth_crop":

                stats["invalid_crop"] += 1

            else:

                stats["errors"] += 1

            continue

        # ----------------------------------------------------
        # Save mouth
        # ----------------------------------------------------

        success = cv2.imwrite(

            str(output_path),

            result["mouth"],

            [
                cv2.IMWRITE_JPEG_QUALITY,
                JPEG_QUALITY,
            ]
        )

        if success:

            stats["mouth_saved"] += 1

        else:

            stats["errors"] += 1

    return stats

In [22]:
# ============================================================
# PHASE 4 — CELL 10
# Production Batch Runner
# ============================================================

ALL_STATS = []
FAILED_VIDEOS = []

print()
print("=" * 70)
print("PHASE 4 — MOUTH CROP BATCH PROCESSING")
print("=" * 70)

print(
    f"Videos found: {len(VIDEO_OUTPUT_DIRS)}"
)

print("=" * 70)


for index, video_output_dir in enumerate(
    VIDEO_OUTPUT_DIRS,
    start=1
):

    video_id = video_output_dir.name

    print()
    print("=" * 70)
    print(
        f"[{index}/{len(VIDEO_OUTPUT_DIRS)}] "
        f"{video_id}"
    )
    print("=" * 70)

    try:

        # ----------------------------------------------------
        # Check Phase 3 input
        # ----------------------------------------------------

        aligned_dir = (
            video_output_dir /
            "aligned_face"
        )

        if not aligned_dir.exists():

            raise FileNotFoundError(
                "Phase 3 aligned_face folder not found:\n"
                f"{aligned_dir}"
            )

        print(
            f"Input : {aligned_dir}"
        )

        print(
            f"Output: "
            f"{video_output_dir / MOUTH_DIR_NAME}"
        )

        # ----------------------------------------------------
        # Process
        # ----------------------------------------------------

        stats = process_video(
            video_output_dir
        )

        ALL_STATS.append(
            stats
        )

        # ----------------------------------------------------
        # Determine status
        # ----------------------------------------------------

        if stats["errors"] > 0:

            status = "COMPLETED_WITH_ERRORS"

        else:

            status = "COMPLETED"

        # ----------------------------------------------------
        # Summary
        # ----------------------------------------------------

        print()
        print(
            f"STATUS         : {status}"
        )

        print(
            f"Video ID       : {stats['video_id']}"
        )

        print(
            f"Input frames   : {stats['input_frames']}"
        )

        print(
            f"Mouth saved    : {stats['mouth_saved']}"
        )

        print(
            f"Skipped        : {stats['skipped']}"
        )

        print(
            f"No landmarks   : {stats['no_landmarks']}"
        )

        print(
            f"Invalid crop   : {stats['invalid_crop']}"
        )

        print(
            f"Errors         : {stats['errors']}"
        )

        # ----------------------------------------------------
        # Failed condition
        # ----------------------------------------------------

        if stats["errors"] > 0:

            FAILED_VIDEOS.append(
                {
                    "video_id":
                        video_id,

                    "reason":
                        "processing_errors",

                    "errors":
                        stats["errors"],
                }
            )

    except Exception as e:

        error_text = (
            f"{type(e).__name__}: {e}"
        )

        print()
        print(
            f"FAILED: {video_id}"
        )

        print(
            f"Error: {error_text}"
        )

        traceback.print_exc()

        FAILED_VIDEOS.append(
            {
                "video_id":
                    video_id,

                "reason":
                    error_text,
            }
        )

    finally:

        gc.collect()


# ============================================================
# Batch Summary
# ============================================================

print()
print("=" * 70)
print("PHASE 4 — BATCH SUMMARY")
print("=" * 70)

if ALL_STATS:

    for stats in ALL_STATS:

        if stats["errors"] > 0:

            status = (
                "COMPLETED_WITH_ERRORS"
            )

        else:

            status = "COMPLETED"

        print(
            f"{stats['video_id']:<15}"
            f"{status:<25}"
            f"mouth={stats['mouth_saved']:<8}"
            f"skip={stats['skipped']:<8}"
            f"error={stats['errors']}"
        )


if FAILED_VIDEOS:

    print()
    print("-" * 70)
    print("VIDEOS REQUIRING ATTENTION")
    print("-" * 70)

    for item in FAILED_VIDEOS:

        print(
            f"{item['video_id']:<15}"
            f"{item['reason']}"
        )

else:

    print()
    print("All videos processed successfully.")


print("=" * 70)


PHASE 4 — MOUTH CROP BATCH PROCESSING
Videos found: 4

[1/4] dataset

FAILED: dataset
Error: FileNotFoundError: Phase 3 aligned_face folder not found:
C:\LipReadingSSL\output\dataset\aligned_face


Traceback (most recent call last):
  File "C:\Users\User\AppData\Local\Temp\ipykernel_3120\3461271966.py", line 49, in <module>
    raise FileNotFoundError(
FileNotFoundError: Phase 3 aligned_face folder not found:
C:\LipReadingSSL\output\dataset\aligned_face



[2/4] video001
Input : C:\LipReadingSSL\output\video001\aligned_face
Output: C:\LipReadingSSL\output\video001\mouth_crop

PHASE 4 — video001


video001:   0%|          | 0/84079 [00:00<?, ?frame/s]

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '



STATUS         : COMPLETED
Video ID       : video001
Input frames   : 84079
Mouth saved    : 83765
Skipped        : 314
No landmarks   : 314
Invalid crop   : 0
Errors         : 0

[3/4] video002
Input : C:\LipReadingSSL\output\video002\aligned_face
Output: C:\LipReadingSSL\output\video002\mouth_crop

PHASE 4 — video002


video002:   0%|          | 0/73701 [00:00<?, ?frame/s]


STATUS         : COMPLETED
Video ID       : video002
Input frames   : 73701
Mouth saved    : 73114
Skipped        : 587
No landmarks   : 587
Invalid crop   : 0
Errors         : 0

[4/4] video003
Input : C:\LipReadingSSL\output\video003\aligned_face
Output: C:\LipReadingSSL\output\video003\mouth_crop

PHASE 4 — video003


video003:   0%|          | 0/28276 [00:00<?, ?frame/s]


STATUS         : COMPLETED
Video ID       : video003
Input frames   : 28276
Mouth saved    : 28107
Skipped        : 169
No landmarks   : 169
Invalid crop   : 0
Errors         : 0

PHASE 4 — BATCH SUMMARY
video001       COMPLETED                mouth=83765   skip=314     error=0
video002       COMPLETED                mouth=73114   skip=587     error=0
video003       COMPLETED                mouth=28107   skip=169     error=0

----------------------------------------------------------------------
VIDEOS REQUIRING ATTENTION
----------------------------------------------------------------------
dataset        FileNotFoundError: Phase 3 aligned_face folder not found:
C:\LipReadingSSL\output\dataset\aligned_face


In [23]:
# ============================================================
# PHASE 4 — CELL 11
# Final Validation
# ============================================================

FINAL_STATS = pd.DataFrame(
    ALL_STATS
)

if FINAL_STATS.empty:

    print(
        "No videos were processed."
    )

else:

    print()
    print("=" * 70)
    print("PHASE 4 — FINAL SUMMARY")
    print("=" * 70)

    print(
        FINAL_STATS.to_string(
            index=False
        )
    )

    print("=" * 70)

    # --------------------------------------------------------
    # Save per-video statistics
    # --------------------------------------------------------

    for row in ALL_STATS:

        video_id = row["video_id"]

        video_output_dir = (
            OUTPUT_DIR /
            video_id
        )

        stats_path = (
            video_output_dir /
            "phase4_stats.csv"
        )

        pd.DataFrame(
            [row]
        ).to_csv(
            stats_path,
            index=False
        )

    # --------------------------------------------------------
    # Verify
    # --------------------------------------------------------

    print()
    print(
        "Verification:"
    )

    for row in ALL_STATS:

        video_id = row["video_id"]

        video_output_dir = (
            OUTPUT_DIR /
            video_id
        )

        mouth_dir = (
            video_output_dir /
            MOUTH_DIR_NAME
        )

        saved = len(
            list(
                mouth_dir.glob(
                    "*.jpg"
                )
            )
        )

        print(
            f"{video_id:<15}"
            f"mouth_crop={saved:<8}"
            f"skipped={row['skipped']:<8}"
            f"errors={row['errors']}"
        )

    print()
    print(
        "Phase 4 completed."
    )


PHASE 4 — FINAL SUMMARY
video_id  input_frames  mouth_saved  skipped  no_landmarks  invalid_crop  errors
video001         84079        83765      314           314             0       0
video002         73701        73114      587           587             0       0
video003         28276        28107      169           169             0       0

Verification:
video001       mouth_crop=83765   skipped=314     errors=0
video002       mouth_crop=73114   skipped=587     errors=0
video003       mouth_crop=28107   skipped=169     errors=0

Phase 4 completed.
